# Simple LangChain Chatbot Tutorial 🤖

---


Welcome to this beginner-friendly tutorial on building a chatbot with LangChain and LangGraph by Drishya!

## What you'll learn:
- How to set up a simple AI chatbot
- Understanding the basic components of LangChain/LangGraph
- How to interact with Google's Gemini AI model
- Building a conversational flow

## Prerequisites:
- No prior Python knowledge required! We'll explain everything step by step
- A Google API key (we'll show you how to get one)

Let's get this started! 🚀


In [1]:
# Install the required packages
# This might take a few minutes the first time you run it

%pip install langchain langgraph langchain-google-genai langchain-tavily

print("✅ Installation complete! Ready to build our chatbot.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.6/426.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.43.0
    Uninstalling google-auth-2.43.0:
      Successfully uninstalled google-auth-2.43.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.55.0
    Uninstalling google-genai-1.55.0:
      Successfully uninstalled google-genai-1.55.0
ERROR: pip's dependency resolver does not currently take into 

✅ Installation complete! Ready to build our chatbot.


In [35]:
import os

# Replace "your_api_key_here" with your actual Google API key
# Make sure to keep the quotes around it!
from google.colab import userdata


# os.environ["GOOGLE_API_KEY"] = "your_api_key_here"
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


# This line checks if the key was set correctly
if os.environ.get("GOOGLE_API_KEY") == "your_api_key_here":
    print("⚠️  Don't forget to replace 'your_api_key_here' with your actual API key!")
else:
    print("✅ API key is set up!")


✅ API key is set up!


In [36]:
# Import statements - these bring in the tools we need
from typing import Annotated                      # Helps with type hints (optional but good practice)
from typing_extensions import TypedDict           # Helps us define data structures

from langgraph.graph import StateGraph, START, END    # The main graph components
from langgraph.graph.message import add_messages      # Handles message management
from langchain.chat_models import init_chat_model     # Connects to AI models

print("✅ All tools imported successfully!")


✅ All tools imported successfully!


In [37]:
# Define what our chatbot will remember
class State(TypedDict):
    # This stores all the messages in our conversation
    # The `add_messages` part tells LangGraph to add new messages to the list
    # instead of replacing the whole list
    messages: Annotated[list, add_messages]

print("✅ State structure defined!")
print("Our chatbot will remember: messages in the conversation")


✅ State structure defined!
Our chatbot will remember: messages in the conversation


In [38]:
# Connect to Google's Gemini AI model
# "gemini-2.0-flash" is a fast and capable version of Google's AI
llm = init_chat_model("google_genai:gemini-2.0-flash")

print("✅ Created a model object of Gemini AI!")
print(f"Model:{llm.model}")


✅ Created a model object of Gemini AI!
Model:gemini-2.0-flash


In [39]:
# This function is the "brain" of our chatbot
def chatbot(state: State):
    """
    This function takes the conversation history and generates a response.

    How it works:
    1. Takes all the messages from the conversation so far
    2. Sends them to the AI model (Gemini)
    3. Gets back a response
    4. Returns the response in the format LangGraph expects
    """
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

print("✅ Chatbot function created!")
print("This function will process messages and generate AI responses.")


✅ Chatbot function created!
This function will process messages and generate AI responses.


In [40]:
# Step 1: Create a graph builder
graph_builder1 = StateGraph(State)

# Step 2: Add our chatbot function as a "node" in the graph
# The first part ("chatbot") is just a name we give it
# The second part (chatbot) is our actual function
graph_builder1.add_node("chatbot", chatbot)

# Step 3: Define the flow: START → chatbot → END
graph_builder1.add_edge(START, "chatbot")  # When conversation starts, go to chatbot
graph_builder1.add_edge("chatbot", END)    # After chatbot responds, end this turn

print("✅ Graph created!")
# Step 4: Build the final graph
graph = graph_builder1.compile()

print("✅ Conversation graph built!")
print("Flow: START → chatbot function → END")



✅ Graph created!
✅ Conversation graph built!
Flow: START → chatbot function → END


In [45]:
# Function to send a message and get a response
def send_message(user_input: str):
    """
    Send a message to the chatbot and display the response.
    """
    print(f"User: {user_input}")

    # Send the message through our graph
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            response = value["messages"][-1].content
            print(f"Assistant: {response}")
    print("-" * 50)  # Separator line

# Test with a simple message
send_message("What is langchain simply?")


User: What is langchain simply?
Assistant: Langchain is essentially a **toolkit for building applications powered by Large Language Models (LLMs)** like GPT-3, PaLM, and others. Think of it as a collection of building blocks and tools that help you:

*   **Connect LLMs to external data sources:** This allows your LLM application to access and use information beyond its initial training data. This could be anything from databases to APIs to documents.

*   **Create complex chains of actions using LLMs:** Instead of just asking a single question, you can orchestrate a series of prompts and actions involving the LLM to accomplish more sophisticated tasks. For example, you could chain together a summarization step with a question-answering step.

*   **Build agents that can reason and act:**  Langchain provides tools to create "agents" that can decide which tools to use (like a search engine, calculator, or database) based on the user's input and then execute those tools to achieve a goal.

In [48]:
# Example 1: Ask about programming
send_message("Can you explain what Python is in simple terms?")


User: Can you explain what Python is in simple terms?
Assistant: Imagine you want to give a robot instructions to do something. You can't just speak to it in English, because it won't understand. You need to use a special language that the robot *does* understand.

Python is like that special language. It's a **programming language** that you use to tell computers what to do.

Here's a breakdown:

*   **It's easy to read and write:**  Python's syntax is designed to be very readable, almost like plain English. This makes it easier to learn and use than some other programming languages.
*   **It's versatile:** You can use Python for lots of different things, like:
    *   **Building websites and web applications:** Think Instagram, Spotify, and YouTube - they use Python in some parts of their systems.
    *   **Analyzing data:** Python is great for crunching numbers and finding insights from large datasets.
    *   **Creating games:** While not the primary language, it's used for game sc

In [50]:
# Example 2: Ask for help with something
send_message("I'm new to AI and chatbots. What should I learn next?")


User: I'm new to AI and chatbots. What should I learn next?
Assistant: That's exciting! The world of AI and chatbots is vast and constantly evolving. Here's a suggested learning path, broken down into categories, with resources and tips to help you navigate:

**I. Foundational Concepts (Essential for everyone):**

*   **What to Learn:**
    *   **Basic AI Concepts:** Understand what AI is (the broad concept of machines mimicking human intelligence), Machine Learning (ML - learning from data without explicit programming), and Deep Learning (DL - a subset of ML using neural networks).
    *   **Natural Language Processing (NLP):** This is crucial for chatbots.  Learn about text analysis, tokenization, part-of-speech tagging, named entity recognition, sentiment analysis, and text generation.
    *   **Chatbot Architecture:**  Grasp the components of a chatbot:
        *   **Natural Language Understanding (NLU):** How the bot interprets user input (intent recognition, entity extraction).
 

In [49]:
# Example 3: Try a creative question
send_message("Write a short poem about robots learning to code")


User: Write a short poem about robots learning to code
Assistant: With circuits humming, gears aligned,
A robot brain, a new design.
No flesh and blood, but metal bright,
It learns to code in digital light.

From simple scripts to complex streams,
It builds a world of coded dreams.
No longer bound by factory walls,
It answers code's compelling calls.

A future blooms, unknown, untold,
As robots write in lines of gold.
And code becomes a language free,
For human minds and circuitry.
--------------------------------------------------


In [ ]:
# Interactive chat session
print("🤖 Welcome to your chatbot!")
print("Type your messages below. Type 'quit' to exit.")
print("=" * 50)

try:
    while True:
        # Get user input
        user_input = input("\nYou: ")

        # Check if user wants to quit
        if user_input.lower() in ["quit", "exit", "q", "bye"]:
            print("Chatbot: Goodbye! Thanks for chatting! 👋")
            break

        # Send message to chatbot
        print("Chatbot: ", end="")
        for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
            for value in event.values():
                response = value["messages"][-1].content
                print(response)

except KeyboardInterrupt:
    print("\n\nChatbot: Goodbye! Thanks for chatting! 👋")
except:
    print("\n\nNote: Interactive input might not work in all environments.")
    print("If you see this message, try using the send_message() function instead!")


🤖 Welcome to your chatbot!
Type your messages below. Type 'quit' to exit.

You: How do RAG systems work?
Chatbot: RAG (Retrieval-Augmented Generation) systems are a powerful approach to building more knowledgeable and accurate AI applications.  They combine the strengths of:

*   **Retrieval:** Finding relevant information from a large external knowledge source (like a document database, website content, etc.).
*   **Generation:** Using a large language model (LLM) to generate a coherent and contextually appropriate answer based on both the user's query *and* the retrieved information.

Here's a step-by-step breakdown of how RAG works:

**1. Indexing (Pre-processing the Knowledge Source):**

*   **Data Preparation:**  The source data (e.g., documents, articles, webpages) is loaded.  This involves cleaning and formatting the text.
*   **Chunking:**  Large documents are split into smaller, more manageable chunks.  The chunk size is crucial:
    *   Too large:  The chunk may contain irrel